In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)

In [4]:
# model declaration
class GatedRecurrentUnit(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_size
        )
        self.input_linear = nn.Linear(
            in_features=embedding_size, out_features=3 * hidden_size
        )
        self.hidden_linear = nn.Linear(
            in_features=hidden_size, out_features=3 * hidden_size
        )
        self.output_linear = nn.Linear(in_features=hidden_size, out_features=vocab_size)

    def forward(self, x, h=None):
        batch_size = x.size(0)

        if h is None:
            h = torch.zeros(size=(batch_size, self.hidden_size))

        logits = []

        e = self.input_linear(self.embedding(x))

        for t in range(x.size(1)):
            xr, xz, xn = e[:, t].chunk(3, dim=-1)
            hr, hz, hn = self.hidden_linear(h).chunk(3, dim=-1)

            # reset: how much of the old h to use when building n
            r = torch.sigmoid(xr + hr)
            # update: how much of the old h to keep
            z = torch.sigmoid(xz + hz)
            # candidate new h
            n = torch.tanh(xn + r * hn)
            h = (1 - z) * n + z * h

            logits.append(self.output_linear(h))

        return torch.stack(tensors=logits, dim=1), h

In [5]:
# model initialization
model = GatedRecurrentUnit(
    vocab_size=vocab_size, embedding_size=embedding_size, hidden_size=hidden_size
)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 241505


In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch]
    y_batch = y_train[batch]
    train_logits, _ = model(x_batch)
    train_loss = nn.functional.cross_entropy(
        input=train_logits.view(-1, vocab_size), target=y_batch.view(-1)
    )

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits, _ = model(x_val)
            val_loss = nn.functional.cross_entropy(
                input=val_logits.view(-1, vocab_size), target=y_val.view(-1)
            )
            print(
                f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}"
            )

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.1954345703125, val loss = 4.193582534790039
Step 1000: train loss = 1.405556559562683, val loss = 1.6112273931503296
Step 2000: train loss = 1.3856247663497925, val loss = 1.5808184146881104
Step 3000: train loss = 1.3466497659683228, val loss = 1.5783077478408813
Step 4000: train loss = 1.3023664951324463, val loss = 1.5741304159164429
Step 4999: train loss = 1.3090006113052368, val loss = 1.5770725011825562


In [7]:
# prediction
with torch.inference_mode():
    output = ["R", "O", "M", "E", "O", ":", "\n"]
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)

    while len(output) < max_output_length:
        logits, h = model(i, h)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [8]:
print("".join(output))

ROMEO:
Her jount, vemompentage thee up didst invointn bear for my knee,
Thou hadst passing me young and noble highor!

ISABELLA:
I cross a means that farewell,
At this be the mouth, and therefore, let, soon mine contrary:
No, with thy month made but of sunsicians broke,
Hadst thou now he is.

POMPEY:
This shall wantone. For Harry, by the battle; there
I came rores of noble Tewrace out: what trow's will
Have we do modinable flattired again to the power
Had none; which for't not a metal spurs:
Will this kindred years on my bridal name;
Yet, as Capuay last,
Who, for your coundry? prove well; fear it, he did swolk.
If I trial for a girl am I
ballad, and decreeting on thee,
By tears true; what's my fellow; is keeps.

ROMEO:
'Tis comfort in the prisoner,
And frew of 'shall be my nublic odd:
Was follow it not spitiff, York all my constant of thy secrets;
And, since that night in all my unborn.

FLORIZEL:
I mistake them despiring
That came the hiuders and brdise married more,
The had cursed in